# LangSmith RAG Tracing Notebook

## Environment Setup

**Kernel:** Python 3.10 (langsmith)

**Conda Environment:** `langsmith`

### Installed Packages:

- langsmith (0.7.3)

- langchain-core (1.2.13)

- langchain-openai (1.1.9)

- langchain-community (0.4.1)

- faiss-cpu (1.13.2)

- openai (2.21.0)

- python-dotenv (1.2.1)

- pydantic (2.12.5)

- numpy (2.2.6)

- requests (2.32.5)

In [ ]:
### To activate this environment manually:
conda activate langsmith

### To verify kernel in Jupyter:

Look for "Python 3.10 (langsmith)" in the kernel selector (top right)

In [ ]:
!pip show langsmith

In [ ]:
# Verify environment setup
import sys
import langsmith
import langchain_core
import langchain_openai

print("Environment Verification")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"Python Executable: {sys.executable}")
print("=" * 60)
print(f"langsmith: {langsmith.__version__}")
print(f"langchain-core: {langchain_core.__version__}")
# print(f"langchain-openai: {langchain_openai.__version__}")
print("=" * 60)
print("Environment is ready for LangSmith tracing!")

!pip show langsmith

In [ ]:
from langsmith import traceable
from openai import OpenAI
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import os

load_dotenv(dotenv_path='../.env')

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Test"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_API_KEY"] = "<Your LangSmith API Key>"  # Update to your API key
# print(os.getenv('LANGSMITH_API_KEY'))
# os.environ["LANGSMITH_API_KEY"]  = "REFER .env file"

client = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

@traceable(
    name="openai_completion",
    run_type="llm",
    tags=["openai", "gpt-4"]
)
def get_completion(prompt: str, model: str = "gpt-4") -> str:
    """Traced OpenAI completion."""

    # LangChain's ChatOpenAI uses .invoke() not .chat.completions.create()
    messages = [
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=prompt)
    ]

    response = client.invoke(messages)
    return response.content

# Usage - automatically traced
result = get_completion("Explain quantum computing in simple terms")
print(result)

## RAG Tacebility

In [ ]:
# Install required packages
!pip install langchain-openai langchain-community langchain-chroma faiss-cpu sentence-transformers

In [ ]:
# Import necessary libraries
import sys
sys.path.append('/Users/vinotganesan/Learning/LLM & AGENTS/RAG')

from langsmith import traceable
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import os
from typing import List

# Import MLServer embeddings - handle import errors
try:
    from ml_server_embedding import get_embeddings
    print("Successfully imported ml_server_embedding")
except Exception as e:
    print(f"Error importing ml_server_embedding: {e}")
    # Fallback - we'll define it inline
    from langchain_core.embeddings import Embeddings
    from pydantic import BaseModel
    import requests
    import numpy as np

    class MLServerEmbedding(Embeddings, BaseModel):
        max_batch_size: int = 32
        url: str

        def embed_documents(self, texts: List[str]) -> List[List[float]]:
            text_num = len(texts)
            result = []
            for i in range(0, text_num, self.max_batch_size):
                result += self._get_embedding(texts[i: i + self.max_batch_size])
            return result

        def embed_query(self, text: str) -> List[float]:
            return self._get_embedding([text])[0]

        @staticmethod
        def _wrap_payload(text_list):
            return {
                "inputs": [
                    {
                        "name": "input",
                        "shape": [len(text_list)],
                        "datatype": "str",
                        "data": text_list
                    }
                ]
            }

        @staticmethod
        def _parse_response(response):
            if response.status_code != 200:
                raise Exception(response)
            outputs = response.json()["outputs"][0]
            return np.array(outputs["data"]).reshape(outputs["shape"]).tolist()

        def _get_embedding(self, text_list: List[str]) -> List[List[float]]:
            return self._parse_response(requests.post(url=self.url,
                                                      json=self._wrap_payload(text_list),
                                                      headers={"Content-Type": "application/json"},
                                                      params={}))

    def get_embeddings(model: str = "all-mpnet-base--fccb5",
                      base_url: str = "",
                      max_batch_size: int = 32):
        embedding_url = f"{base_url}/{model}/v2/models/{model}/infer"
        return MLServerEmbedding(url=embedding_url, max_batch_size=max_batch_size)

    print("Using inline MLServerEmbedding class")

# Load environment variables
load_dotenv()

# Configure LangSmith tracing
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "RAG_Tracing_Demo"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGSMITH_API_KEY"]  = "REFER .env file"

# Configure LLM
api_key = os.environ['UNIFIED_LLM_KEY']
base_url = ""
print(f'API key configured: {api_key[:20]}...')

In [ ]:
# Create sample documents for RAG
documents = [
    Document(
        page_content="LangSmith is a platform for building production-grade LLM applications. It provides debugging, testing, evaluating, and monitoring capabilities.",
        metadata={"source": "langsmith_docs", "page": 1}
    ),
    Document(
        page_content="LangSmith tracing allows you to log and visualize your LLM application runs. It captures inputs, outputs, latency, and token usage.",
        metadata={"source": "langsmith_docs", "page": 2}
    ),
    Document(
        page_content="RAG (Retrieval Augmented Generation) is a technique that combines retrieval of relevant documents with LLM generation to provide more accurate and contextual responses.",
        metadata={"source": "rag_docs", "page": 1}
    ),
    Document(
        page_content="Vector databases store embeddings of documents and enable semantic search. Popular options include FAISS, Pinecone, Chroma, and Weaviate.",
        metadata={"source": "vector_db_docs", "page": 1}
    ),
    Document(
        page_content="LangChain is a framework for developing applications powered by language models. It provides tools for chaining together LLM calls, memory, and data sources.",
        metadata={"source": "langchain_docs", "page": 1}
    )
]

print(f"Created {len(documents)} sample documents")

In [ ]:
# Create embeddings and vector store with error handling
@traceable(name="create_vector_store", run_type="tool", tags=["embeddings", "vector_store"])
def create_vector_store(docs):
    """Create FAISS vector store from documents with tracing."""
    try:
        print("Creating embeddings instance...")
        # Use MLServer embeddings from ml_server_embedding.py
        embeddings = get_embeddings()  # Uses default model: all-mpnet-base--fccb5
        print(f"Embeddings instance created: {type(embeddings)}")

        print(f"Creating vector store from {len(docs)} documents...")
        vectorstore = FAISS.from_documents(docs, embeddings)
        print(f"Vector store created successfully with {vectorstore.index.ntotal} vectors")

        return vectorstore
    except Exception as e:
        print(f"Error creating vector store: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        raise

# Create the vector store
try:
    vectorstore = create_vector_store(documents)
    print("\nVector store ready for retrieval!")
except Exception as e:
    print(f"\nFailed to create vector store: {e}")
    vectorstore = None

In [ ]:
# Create traced retriever function
@traceable(name="retrieve_documents", run_type="retriever", tags=["retrieval", "similarity_search"])
def retrieve_documents(query: str, k: int = 3):
    """Retrieve relevant documents with tracing."""
    if vectorstore is None:
        raise ValueError("Vector store not initialized. Please run the vector store creation cell first.")

    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)

    # Log retrieved document details
    doc_details = [
        {
            "content": doc.page_content[:100] + "...",
            "metadata": doc.metadata
        }
        for doc in docs
    ]
    print(f"Retrieved {len(docs)} documents")
    return docs

In [ ]:
# Test retrieval only if vectorstore is ready
if vectorstore is not None:
    test_query = "What is LangSmith tracing?"
    retrieved_docs = retrieve_documents(test_query)
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\nDocument {i}:")
        print(f"Content: {doc.page_content[:150]}...")
        print(f"Source: {doc.metadata.get('source', 'N/A')}")
else:
    print("Skipping retrieval test - vector store not ready")

In [ ]:
# Create RAG chain with full tracing
@traceable(name="format_context", run_type="tool", tags=["formatting"])
def format_docs(docs):
    """Format retrieved documents into context string."""
    return "\n\n".join([doc.page_content for doc in docs])

# Create prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI assistant. Use the following context to answer the user's question.
If you cannot answer based on the context, say so.

Context:
{context}"""),
    ("human", "{question}")
])

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=api_key,
    base_url=base_url
)

# Create the RAG chain
rag_chain = (
    {"context": lambda x: format_docs(retrieve_documents(x)), "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully!")

In [ ]:
# Complete traced RAG function
@traceable(
    name="rag_query",
    run_type="chain",
    tags=["rag", "production"],
    metadata={"version": "1.0", "model": "gpt-4o-mini"}
)
def rag_query(question: str) -> str:
    """
    Complete RAG pipeline with tracing.
    This will trace the entire RAG flow in LangSmith.
    """
    response = rag_chain.invoke(question)
    return response

# Test the RAG system with multiple queries
test_questions = [
    "What is LangSmith and what features does it provide?",
    "Explain RAG and how it works",
    "What are some popular vector databases?"
]

print("Running RAG queries with LangSmith tracing...\n")
print("=" * 80)

for i, question in enumerate(test_questions, 1):
    print(f"\nQuery {i}: {question}")
    print("-" * 80)
    answer = rag_query(question)
    print(f"Answer: {answer}")
    print("=" * 80)

## Advanced RAG Tracing with Custom Metadata

In [ ]:
from langsmith import trace
from typing import Dict, List, Any
import time

class TracedRAGPipeline:
    """
    Advanced RAG pipeline with comprehensive LangSmith tracing.
    Tracks retrieval quality, latency, and custom metrics.
    """

    def __init__(self, vectorstore, llm):
        self.vectorstore = vectorstore
        self.llm = llm

    @traceable(
        name="enhanced_retrieval",
        run_type="retriever",
        tags=["retrieval", "similarity", "metadata"]
    )
    def retrieve_with_scores(self, query: str, k: int = 3) -> List[tuple]:
        """Retrieve documents with similarity scores."""
        start_time = time.time()

        # Get documents with scores
        docs_with_scores = self.vectorstore.similarity_search_with_score(query, k=k)

        retrieval_time = time.time() - start_time

        # Log custom metrics
        metrics = {
            "num_retrieved": len(docs_with_scores),
            "retrieval_time_ms": retrieval_time * 1000,
            "avg_score": sum(score for _, score in docs_with_scores) / len(docs_with_scores) if docs_with_scores else 0
        }

        print(f"Retrieval metrics: {metrics}")
        return docs_with_scores

    @traceable(
        name="rerank_documents",
        run_type="tool",
        tags=["reranking", "filtering"]
    )
    def rerank_documents(self, docs_with_scores: List[tuple], threshold: float = 0.5) -> List[Document]:
        """Filter and rerank documents based on score threshold."""
        filtered_docs = [
            doc for doc, score in docs_with_scores
            if score <= threshold  # Lower score is better in FAISS
        ]

        print(f"Filtered {len(docs_with_scores)} -> {len(filtered_docs)} documents (threshold: {threshold})")
        return filtered_docs

    @traceable(
        name="generate_answer",
        run_type="llm",
        tags=["generation", "llm_call"]
    )
    def generate_answer(self, query: str, context_docs: List[Document]) -> Dict[str, Any]:
        """Generate answer with context and metadata."""
        start_time = time.time()

        # Format context
        context = "\n\n".join([doc.page_content for doc in context_docs])

        # Create prompt
        messages = [
            ("system", f"""You are a helpful AI assistant. Use the following context to answer the question.
If you cannot answer based on the context, say so.

Context:
{context}"""),
            ("human", query)
        ]

        prompt = ChatPromptTemplate.from_messages(messages)
        chain = prompt | self.llm | StrOutputParser()

        # Generate response
        answer = chain.invoke({"question": query})

        generation_time = time.time() - start_time

        return {
            "answer": answer,
            "num_context_docs": len(context_docs),
            "generation_time_ms": generation_time * 1000,
            "context_length": len(context)
        }

    @traceable(
        name="rag_pipeline_full",
        run_type="chain",
        tags=["rag", "production", "full_pipeline"],
        metadata={"pipeline_version": "2.0"}
    )
    def query(self, question: str, k: int = 3, score_threshold: float = 0.5) -> Dict[str, Any]:
        """
        Complete RAG pipeline with full tracing and metrics.

        Args:
            question: User query
            k: Number of documents to retrieve
            score_threshold: Similarity score threshold for filtering

        Returns:
            Dictionary with answer and metadata
        """
        pipeline_start = time.time()

        # Step 1: Retrieve documents with scores
        docs_with_scores = self.retrieve_with_scores(question, k=k)

        # Step 2: Rerank/filter documents
        filtered_docs = self.rerank_documents(docs_with_scores, threshold=score_threshold)

        # Step 3: Generate answer
        if not filtered_docs:
            return {
                "answer": "No relevant documents found to answer the question.",
                "num_retrieved": 0,
                "total_time_ms": (time.time() - pipeline_start) * 1000
            }

        result = self.generate_answer(question, filtered_docs)

        # Add pipeline-level metrics
        result["total_time_ms"] = (time.time() - pipeline_start) * 1000
        result["num_retrieved"] = len(docs_with_scores)
        result["num_used"] = len(filtered_docs)

        return result

# Initialize the traced RAG pipeline
traced_pipeline = TracedRAGPipeline(vectorstore, llm)
print("Advanced RAG pipeline initialized!")

In [ ]:
# Test the advanced RAG pipeline with detailed tracing
test_queries = [
    "What capabilities does LangSmith provide for LLM applications?",
    "How does RAG improve LLM responses?",
    "What is the purpose of vector databases in RAG systems?"
]

print("\n" + "="*100)
print("TESTING ADVANCED RAG PIPELINE WITH LANGSMITH TRACING")
print("="*100 + "\n")

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*100}")
    print(f"Query {i}: {query}")
    print('='*100)

    # Run query with full tracing
    result = traced_pipeline.query(query, k=3, score_threshold=1.0)

    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nMetrics:")
    print(f"   - Total Time: {result['total_time_ms']:.2f}ms")
    print(f"   - Documents Retrieved: {result['num_retrieved']}")
    # print(f"   - Documents Used: {result['num_used']}")
    if 'generation_time_ms' in result:
        print(f"   - Generation Time: {result['generation_time_ms']:.2f}ms")
        print(f"   - Context Length: {result['context_length']} chars")
    print(f"\n{'='*100}\n")

## Collecting User Feedback

Track user satisfaction and ratings by associating feedback with specific runs. This allows you to measure the quality of your RAG system based on actual user feedback.

In [ ]:
from langsmith import uuid7, Client

# Initialize LangSmith client
ls_client = Client()

# Generate unique run ID for tracking
run_id = str(uuid7())
print(f"Generated Run ID: {run_id}")

# Execute RAG query with run ID - using a question relevant to our documents
question = "What features and capabilities does LangSmith provide for building LLM applications?"
print(f"\nQuestion: {question}")

result = rag_query(
    question,
    langsmith_extra={"run_id": run_id}
)

print(f"Answer: {result}")
print(f"\nRun ID for feedback: {run_id}")

In [ ]:
# Collect positive user feedback
print("Submitting positive feedback...")

# Add user score (1.0 for positive, 0.0 for negative)
ls_client.create_feedback(
    run_id=run_id,
    key="user-score",
    score=1.0,
    comment="Comprehensive and accurate response about LangSmith features"
)

print("Score feedback submitted (1.0 - positive)")

# Add detailed user comment
ls_client.create_feedback(
    run_id=run_id,
    key="user-comment",
    comment="The answer clearly explained all key features: debugging, testing, evaluating, and monitoring. It also mentioned tracing capabilities which was very helpful!"
)

print("Comment feedback submitted")
print(f"\nView feedback in LangSmith: https://smith.langchain.com")
print(f"Feedback helps track RAG quality and identify areas for improvement")

In [ ]:
# Example: Collect negative feedback for a question outside the knowledge base
print("Example of negative feedback...")

# Generate another run ID for negative example
run_id_negative = str(uuid7())

# Ask a question that's NOT in our document set
negative_question = "What is the pricing model for LangSmith enterprise plans?"
print(f"\nQuestion: {negative_question}")

result_negative = rag_query(
    negative_question,
    langsmith_extra={"run_id": run_id_negative}
)

print(f"Answer: {result_negative[:150]}...")

# Submit negative feedback because the answer lacks specific pricing info
ls_client.create_feedback(
    run_id=run_id_negative,
    key="user-score",
    score=0.0,  # 0.0 for negative
    comment="Response did not provide pricing information"
)

ls_client.create_feedback(
    run_id=run_id_negative,
    key="user-comment",
    comment="The answer did not address the pricing question. Need to add pricing documentation to the knowledge base."
)

print("\nNegative feedback submitted (score: 0.0)")
print(f"Run ID: {run_id_negative}")
print("\nTip: Use feedback scores to track RAG system quality over time")
print("Negative feedback helps identify gaps in the knowledge base")

### Benefits of User Feedback

- **Quality Measurement**: Track user satisfaction over time

- **Issue Identification**: Find problematic queries or responses

- **Model Improvement**: Use feedback to fine-tune or improve prompts

- **A/B Testing**: Compare different versions based on user feedback

- **Analytics**: View feedback trends in LangSmith Monitor tab

## Production Monitoring & A/B Testing

Monitor production metrics and compare different model versions using LangSmith's built-in analytics.

### Key Metrics in Production

When monitoring your RAG system in production, LangSmith tracks:

1. **Trace Volume**: Number of traces over time

2. **Latency**: P50, P95, P99 latency percentiles

3. **Token Usage**: Input/output token consumption

4. **Feedback Metrics**: User ratings and satisfaction scores

5. **Error Rates**: Failed runs and exceptions

6. **Cost Tracking**: Estimated costs based on token usage

In [ ]:
# A/B Testing: Version A - Using GPT-4o-mini
@traceable(
    name="rag_query_v1",
    run_type="chain",
    tags=["rag", "production", "experiment_a"],
    metadata={"model_version": "gpt-4o-mini", "experiment": "A", "prompt_version": "v1.0"}
)
def rag_query_v1(question: str) -> str:
    """RAG Version A: GPT-4o-mini with standard prompt."""
    llm_v1 = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=api_key,
        base_url=base_url
    )

    rag_chain_v1 = (
        {"context": lambda x: format_docs(retrieve_documents(x)), "question": RunnablePassthrough()}
        | prompt_template
        | llm_v1
        | StrOutputParser()
    )

    return rag_chain_v1.invoke(question)

print("Version A (GPT-4o-mini) defined")

In [ ]:
# A/B Testing: Version B - Using GPT-4 with enhanced prompt
@traceable(
    name="rag_query_v2",
    run_type="chain",
    tags=["rag", "production", "experiment_b"],
    metadata={"model_version": "gpt-4", "experiment": "B", "prompt_version": "v2.0"}
)
def rag_query_v2(question: str) -> str:
    """RAG Version B: GPT-4 with enhanced prompt."""
    llm_v2 = ChatOpenAI(
        model="gpt-4o",
        temperature=0.2,  # Slightly higher temperature for more diverse responses
        api_key=api_key,
        base_url=base_url
    )

    # Enhanced prompt template
    enhanced_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert AI assistant. Use the following context to provide a comprehensive answer.
Be specific, accurate, and cite relevant information from the context.

Context:
{context}"""),
        ("human", "{question}")
    ])

    rag_chain_v2 = (
        {"context": lambda x: format_docs(retrieve_documents(x)), "question": RunnablePassthrough()}
        | enhanced_prompt
        | llm_v2
        | StrOutputParser()
    )

    return rag_chain_v2.invoke(question)

print("Version B (GPT-4 with enhanced prompt) defined")

In [ ]:
# Run A/B test with sample queries
import random

test_queries_ab = [
    "What is LangSmith?",
    "How does RAG work?",
    "What are vector databases used for?",
    "Explain LangChain framework",
    "What tracing capabilities does LangSmith provide?"
]

print("Running A/B Test with 5 queries...")
print("=" * 80)

for i, query in enumerate(test_queries_ab, 1):
    # Randomly assign to version A or B (50/50 split)
    version = random.choice(['A', 'B'])

    print(f"\nQuery {i}: {query}")
    print(f"Assigned to: Version {version}")
    print("-" * 80)

    if version == 'A':
        answer = rag_query_v1(query)
        print(f"Version A Response:\n{answer[:200]}...")
    else:
        answer = rag_query_v2(query)
        print(f"Version B Response:\n{answer[:200]}...")

    print("=" * 80)

print("\nA/B Test completed!")
print("View results in LangSmith Monitor tab")
print("Filter by 'experiment' metadata to compare A vs B")

### How to View A/B Test Results in LangSmith

1. **Navigate to LangSmith**: Go to https://smith.langchain.com

2. **Select Your Project**: Click on "RAG_Tracing_Demo"

3. **Open Monitor Tab**: Click the "Monitor" tab

4. **Filter by Metadata**:

   - Click the **Metadata** button on any chart

   - Select `experiment` as the metadata field

   - Compare metrics between experiments A and B

### Metrics to Compare:

- **Latency**: Which version is faster?

- **Token Usage**: Which version is more cost-effective?

- **Feedback Scores**: Which version gets better user ratings?

- **Error Rates**: Which version is more reliable?

### Example Analysis:

In [ ]:
Version A (GPT-4o-mini):
- Faster response time
- Lower token cost
- Good for high-volume queries

Version B (GPT-4 enhanced):
- Higher quality responses
- More detailed answers
- Better for complex queries

### Best Practices for Production Monitoring

In [ ]:
1. **Use Meaningful Project Names**
   os.environ["LANGSMITH_PROJECT"] = "production-rag-chatbot"

In [ ]:
2. **Add Contextual Metadata**
   langsmith_extra={
       "metadata": {
           "user_tier": "premium",
           "feature_flag": "new_retrieval_v2",
           "region": "us-east-1"
       }
   }

In [ ]:
3. **Track Run IDs for Feedback**
   # Always generate and store run IDs when user interaction is involved
   run_id = str(uuid7())
   result = rag_query(question, langsmith_extra={"run_id": run_id})
   # Store run_id for later feedback collection

4. **Monitor Production Metrics**

   - Set up alerts for latency thresholds

   - Track feedback trends over time

   - Compare performance across model versions

   - Monitor token costs and usage patterns

5. **Drilldown and Debugging**

   - Navigate to Monitor tab in LangSmith

   - Hover over data points in charts (e.g., high latency spike)

   - Click to filter runs table to that time period

   - Review individual traces to identify root causes

## Summary

This notebook demonstrates comprehensive LangSmith tracing for RAG applications:

### What We Covered:

1. **Basic Tracing**

   - Environment setup

   - Simple LLM call tracing

   - Automatic trace logging

2. **RAG Pipeline Tracing**

   - Document retrieval tracing

   - Vector store operations

   - End-to-end RAG chain tracing

   - Custom metadata and tags

3. **Advanced Tracing**

   - Performance metrics tracking

   - Document reranking

   - Custom run types

   - Nested trace hierarchy

4. **User Feedback Collection**

   - Generating unique run IDs

   - Collecting positive/negative feedback

   - Adding comments and scores

   - Tracking user satisfaction

5. **Production Monitoring & A/B Testing**

   - Defining multiple model versions

   - Comparing performance metrics

   - Tracking experiments

   - Production analytics

### View Your Traces:

Go to [LangSmith](https://smith.langchain.com) and navigate to:

- **Project**: RAG_Tracing_Demo

- **Monitor Tab**: View aggregated metrics

- **Runs Tab**: Drill down into individual traces

### Additional Resources:

- [LangSmith Documentation](https://docs.langchain.com/langsmith)

- [API Reference](https://api.python.langchain.com/en/latest/langsmith_api_reference.html)

- [LangSmith Playbook](./langsmith_playbook.md)